# Homework 12.5 - Coding

This is the coding portion of the homework assignment for Section 12.5

In [42]:
from typing import Callable
import numpy as np
from jax import numpy as jnp
from jax import grad

## Problem 12.30

### Part (0)

Consider implementation of the BFGS algorithm provided below, adapted from the textbook to be more in line with similar methods coded up in previous homeworks (that is, adapted to use automatic
differentiation software to compute the gradient $Df$ rather than providing it as an argument to the
BFGS function, as in the textbook).

In this updated implementation, demonstrate your understanding of the algorithm by doing the following:

1. Finish the docstring by providing a short, less-than-1-sentence description of what each of the 5 input arguments/parameters actually is (indicated by the "TODO" statements in the docstring)
2. In the code itself, replace any comments with `TODO:` written in in them with the requested material in a one-line, succinct comment.

In [43]:
def BFGS(
    f: Callable[[jnp.ndarray], float],
    x0: jnp.ndarray,
    A0: jnp.ndarray,
    tol: float = 1e-8,
    maxiter: int = 40,
) -> tuple[jnp.ndarray, int, bool]:
    """Approximates the minimizer of a function x using
    the BFGS low-rank-update method.

    This method approximates the minimizer of f in a computationally efficient
    way by successively updating an approximation of the Hessian of the function f (A_k) at 
    each iteration, in addition to successively updating an approximation of the 
    minimizer of the function f (x_k) at each iteration.

    After being given initial guesses x0 for the minimizer and A0 for the Hessian,
    updates take the form:
        1. Aproximations of minimizer - Equation (12.25) in book:
            x_{k+1} = x_k - (A_k)^{-1} Df(x_k)^T    
        2. Approximations of (inverse of) Hessian - Equation (12.33) in book: 
            (A_{k+1})^{-1} =
                (A_k)^{-1} 
                + (<s_{k+1}, y_{k+1}> + <y_{k+1}, (A_k)^{-1} y_{k+1}>)/(<s_{k+1}, y_{k+1}>)^2) s_{k+1} s_{k+1}^T
                - [(A_k)^{-1} y_{k+1} s_{k+1}^T + s_{k+1} [(A_k)^{-1} y_{k+1}]^T / <s_{k+1}, y_{k+1}>
            where < a, b > denotes the standard inner/dot product of a and b on R^n,
            (a b^T) denotes the standard outer prouct of a and b on R^n, 
            and 
                s_{k+1} := x_{k+1} - x_k                (difference in position)  - Equation (12.30) in book
                y_{k+1} := Df(x_{k+1})^T - Df(x_k)^T    (difference in gradient)  - Equation (12.30) in book
            are the needed vectors in this update formula.

    Args:
        f (Callable): The objective function
        x0 (jnp.ndarray): initial guess for x.
        A0 (jnp.ndarray): initial guess for A0.
        tol (float): tolerance of the function.  When any of the stopping critera given this tol are met, the function terminates
        maxiter (int): max number of iterations.

    Returns:
        jnp.ndarray - An approximation to the minimizer of f 
        int - The number of iterations (indexed by k in the book) that the algorithm ran
        bool - Whether or not the BFGS algorithm converged in the given number of iterations
    """
    # compute Df and A^-1
    Df = grad(f)
    A_inv = jnp.linalg.inv(A0)

    # begin the loop by setting x_k = x0, and iterate AT MOST maxiter times
    x_k = x0
    for i in range(maxiter):
        x_k_plus1 = x_k - (A_inv @ Df(x_k))     # update x_k_plus1 with equations 12.25.  This is the new point on the surface.  A_inv @ Df(x_k) is the direction to travel in
        s_k_plus1 = x_k_plus1 - x_k             # update s_k_plus1 with equation 12.30.  This is the step size
        y_k_plus1 = Df(x_k_plus1) - Df(x_k)     # the change in gradient

        # if any of the convergence criteria are met
        if (
            (jnp.linalg.norm(s_k_plus1) < tol) or           # convergence criterion: step size is within tol
            (jnp.linalg.norm(Df(x_k_plus1)) < tol) or       # convergence criterion: derivative is within tol, meaning it's quite flat and therefore close to the minimizer
            (jnp.abs(f(x_k_plus1) - f(x_k)) < tol)          # convergence criterion: consecutive points in the interation are within tol of eachother
        ):
            return x_k_plus1, i+1, True
        
        # put together the pieces in (12.29)
        sy = s_k_plus1 @ y_k_plus1
        Ay = A_inv @ y_k_plus1 
        A_inv += (
            ((sy + y_k_plus1 @ Ay)/sy**2) * jnp.outer(s_k_plus1, s_k_plus1)
            - (jnp.outer(Ay, s_k_plus1) + jnp.outer(s_k_plus1, Ay)) / sy
        )

        # Set up for next iteration 
        x_k = x_k_plus1
    
    # if we get here then we hit all of the iterations in maxiter, so return False for if it converged
    return x_k, maxiter, False

### Part (i)

Now, apply the above code to the function in Example 12.5.3; that is, use the `BFGS()` function from above to attempt to find the minimizer of
$$f(x,y) = x^3 - 3x^2 + y^2$$
which has a local minimizer $\mathbf{x}^* = (x^*, y^*) = (2,0)$

Seed the BFGS algorithm with an initial guess of $\mathbf{x}_0 = (x_0, y_0) = (4,4)$ for the minimizer and $A_0 = D^2f(\mathbf{x}_0) = \begin{bmatrix} 18 & 0 \\ 0 & 2 \end{bmatrix}$ for the Hessian.

Keep trying your code with various error tolerances/numbers of iterations until you can get your algorithm to converge to within $10^{-5}$ of the true solution $\mathbf{x}^* = (x^*, y^*) = (2,0)$.

In [44]:
f = lambda x: x[0]**3 - 3*x[0]**2 + x[1]**2
x0 = jnp.array([4., 4.])
A0 = jnp.array([
    [18., 0.],
    [0., 2.]
])
tol = 0.0001       # Play around with this parameter 
maxiter = 10     # And this one 

x, maxiter, converged = BFGS(f, x0, A0, tol, maxiter)
print(f"x_k: {x}")
print(f"converged: {converged}")
print(f"maxiter: {maxiter}")

x_k: [2.0000014e+00 5.7572324e-06]
converged: True
maxiter: 6


How many iterations does your code take to get the approximate minimizer within $10^{-5}$ of the true minimizer $(2,0)$? What error tolerance did you need to use on the BFGS algorithm?

**RESPONSE:** _Around tol = 0.0001, maxiter=6._

### Part (ii)

Repeat part (i) with new initial guesses of $\mathbf{x}_0 = (x_0, y_0) = (4,4)$ for the minimizer and $A_0 = I = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}$ for the Hessian, playing with your tolerance/maximum number of iterations until you get an approximation within $10^{-5}$ of the true minimizer $\mathbf{x}^* = (2,0)$

In [45]:
f = lambda x: x[0]**3 - 3*x[0]**2 + x[1]**2
x0 = jnp.array([4., 4.])
A0 = jnp.eye(2)
tol = 0.00001       # Play around with this parameter 
maxiter = 100     # And this one 

x, maxiter, converged = BFGS(f, x0, A0, tol, maxiter)
print(f"x_k: {x}")
print(f"converged: {converged}")
print(f"maxiter: {maxiter}")

x_k: [ 1.999984e+00 -1.019350e-05]
converged: True
maxiter: 14


How many iterations does your code take to get the approximate minimizer within $10^{-5}$ of the true minimizer $(2,0)$? What error tolerance did you need to use on the BFGS algorithm?

**RESPONSE:** _Around tol=0.00001 and maxiter=14._

### Part (iii)

Repeat part (i) with new initial guesses of $\mathbf{x}_0 = (x_0, y_0) = (10,10)$ for the minimizer and $A_0 = D^2f(\mathbf{x}_0) = \begin{bmatrix} 54 & 0 \\ 0 & 2 \end{bmatrix}$ for the Hessian, playing with your tolerance/maximum number of iterations until you get an approximation within $10^{-5}$ of the true minimizer $\mathbf{x}^* = (2,0)$

In [46]:
f = lambda x: x[0]**3 - 3*x[0]**2 + x[1]**2
x0 = jnp.array([10., 10.])
A0 = jnp.array([
    [54., 0.],
    [0., 2.]
])
tol = 0.01       # Play around with this parameter 
maxiter = 7     # And this one 

x, maxiter, converged = BFGS(f, x0, A0, tol, maxiter)
print(f"x_k: {x}")
print(f"converged: {converged}")
print(f"maxiter: {maxiter}")

x_k: [ 2.0010424e+00 -2.9277089e-06]
converged: True
maxiter: 7


How many iterations does your code take to get the approximate minimizer within $10^{-5}$ of the true minimizer $(2,0)$? What error tolerance did you need to use on the BFGS algorithm?

**RESPONSE:** _TODO: Around tol=0.01, maxiter = 7._

### Part (iv)

Repeat part (i) with new initial guesses of $\mathbf{x}_0 = (x_0, y_0) = (10,10)$ for the minimizer and $A_0 = I = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}$ for the Hessian, playing with your tolerance/maximum number of iterations until you get an approximation within $10^{-5}$ of the true minimizer $\mathbf{x}^* = (2,0)$.

In [47]:
f = lambda x: x[0]**3 - 3*x[0]**2 + x[1]**2
x0 = jnp.array([10., 10.])
A0 = jnp.eye(2)
tol = 0.00001       # Play around with this parameter 
maxiter = 20     # And this one 

x, maxiter, converged = BFGS(f, x0, A0, tol, maxiter)
print(f"x_k: {x}")
print(f"converged: {converged}")
print(f"maxiter: {maxiter}")

x_k: [ 1.9999441e+00 -7.3629548e-05]
converged: True
maxiter: 17


How many iterations does your code take to get the approximate minimizer within $10^{-5}$ of the true minimizer $(2,0)$? What error tolerance did you need to use on the BFGS algorithm?

**RESPONSE:** _Around tol=0.00001, maxiter=20._

### Part (v)

Try running your code $\mathbf{x}_0 = (0,0)$, with any given initial guess $A_0$ for the Hessian (the identity is probably the easiest here).

In [48]:
f = lambda x: x[0]**3 - 3*x[0]**2 + x[1]**2
x0 = jnp.array([0., 0.])
A0 = jnp.eye(2)
tol = 0.0001
maxiter = 5

x, maxiter, converged = BFGS(f, x0, A0, tol, maxiter)
print(f"x_k: {x}")
print(f"converged: {converged}")
print(f"maxiter: {maxiter}")

x_k: [0. 0.]
converged: True
maxiter: 1


Does this converge to $\mathbf{x}^* = (2,0)$? If so, how many iterations does it take? If not, why does the algorithm fail to converge?

**RESPONSE:** _No, because it starts on a saddle point so the first derivative is zero so the algorithm stops.._

---

IMPORTANT: Please "Restart and Run All" and ensure there are no errors. Then, submit this .ipynb file to Gradescope.